# Data Analysis 

LSE ID = 

## Loading the tables from the database

First, I start importing all the relevant libraries and loading the csv files I'll use in this notebook

In [33]:
import pandas as pd
import plotly.express as px
import sqlite3

In [34]:
conn = sqlite3.connect("../data/movies.db")

In [35]:
movies = pd.read_sql("SELECT * FROM movies", conn)
genres = pd.read_sql("SELECT * FROM genres", conn)
genres_movie = pd.read_sql("SELECT * FROM movie_genres", conn)



## Analysis 1: Distribution of runtime per year

First, I group the movies dataframe by year. Then, for each year, I take the mean for runtime.

In [36]:
fig= px.histogram(movies, x="runtime" , facet_row="year", width=500, height=2000)
fig.update_yaxes(title_text="Number of movies")


In [41]:
genres_movie

,id,genre_id,genre_name


In [ ]:
fig= px.histogram(movies, x="runtime" , facet_row="genre", width=500, height=2000)
fig.update_yaxes(title_text="Number of movies")

## Analysis 2: What genres dominated the 2010's vs the 2020's 

For this analysis, I'll use a query. This query groups rows in terms of year and genre_name. Then for each group, it counts the total movies. The query then orders it by year (with the genres with most movies appearing at the top for each year)

In [115]:
q1_df = pd.read_sql("""

SELECT 
    m.year,
    g.genre_name,
    COUNT(m.id) AS total_movies
FROM movies m
JOIN movie_genres g ON m.id = g.id
GROUP BY m.year, g.genre_name
ORDER BY m.year ASC, total_movies DESC;

""", conn) 

q1_df

,year,genre_name,total_movies
0,2015,Action,40
1,2015,Comedy,38
2,2015,Adventure,38
3,2015,Drama,33
4,2015,Thriller,23
...,...,...,...
179,2025,Romance,10
180,2025,War,8
181,2025,History,8
182,2025,Music,4


Since I don't want the plot to be difficult to analyze, I take the query output (dataframe), and then take the top 3 genres with most movies (across all years). After that, I make a mask that only filter those specific genres

In [116]:
genres_plot = q1_df.groupby("genre_name")["total_movies"].sum().nlargest(3).index

q1_df = q1_df[q1_df["genre_name"].isin(genres_plot)]

In [117]:
q1_df

,year,genre_name,total_movies
0,2015,Action,40
1,2015,Comedy,38
2,2015,Adventure,38
17,2016,Adventure,45
18,2016,Action,44
19,2016,Comedy,33
34,2017,Action,45
36,2017,Adventure,33
37,2017,Comedy,31
50,2018,Action,43


In [118]:
px.line(q1_df, x="year", y="total_movies", color="genre_name", markers = True) 

## Analysis 3

In [119]:
q2_df = pd.merge(movies, genres_movie, on = "id", how = "inner")

In [120]:
q2_df

,id,title,original_title,release_date,budget,revenue,popularity,vote_average,vote_count,runtime,original_language,year,genre_id,genre_name
0,140607,Star Wars: The Force Awakens,Star Wars: The Force Awakens,2015-12-15,245000000,2068223624,21.4627,7.248,20687,136,en,2015,12,Adventure
1,140607,Star Wars: The Force Awakens,Star Wars: The Force Awakens,2015-12-15,245000000,2068223624,21.4627,7.248,20687,136,en,2015,28,Action
2,140607,Star Wars: The Force Awakens,Star Wars: The Force Awakens,2015-12-15,245000000,2068223624,21.4627,7.248,20687,136,en,2015,878,Science Fiction
3,135397,Jurassic World,Jurassic World,2015-06-06,150000000,1671537444,18.7528,6.703,21726,124,en,2015,12,Adventure
4,135397,Jurassic World,Jurassic World,2015-06-06,150000000,1671537444,18.7528,6.703,21726,124,en,2015,878,Science Fiction
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3186,1233575,Black Bag,Black Bag,2025-03-12,50000000,43887905,6.7802,6.367,1294,94,en,2025,9648,Mystery
3187,1233575,Black Bag,Black Bag,2025-03-12,50000000,43887905,6.7802,6.367,1294,94,en,2025,53,Thriller
3188,701387,Bugonia,Bugonia,2025-10-23,55000000,43500000,19.2877,7.304,2161,119,en,2025,878,Science Fiction
3189,701387,Bugonia,Bugonia,2025-10-23,55000000,43500000,19.2877,7.304,2161,119,en,2025,53,Thriller


In [121]:
q2_df[["title", "genre_name", "runtime"]]

,title,genre_name,runtime
0,Star Wars: The Force Awakens,Adventure,136
1,Star Wars: The Force Awakens,Action,136
2,Star Wars: The Force Awakens,Science Fiction,136
3,Jurassic World,Adventure,124
4,Jurassic World,Science Fiction,124
...,...,...,...
3186,Black Bag,Mystery,94
3187,Black Bag,Thriller,94
3188,Bugonia,Science Fiction,119
3189,Bugonia,Thriller,119


In [122]:
px.box(q2_df, x="genre_name", y="runtime", color = "genre_name", points = "outliers")